In [ ]:
import os

os.environ["GPT_API_KEY"] = "sk-"

from convokitai import Corpus
from convokitai_evaluation import ClaimTracker

corpus = Corpus.load("sample_corpus")

In [2]:
# downloads the Qwen3 embedding and DeBERTa NLI models on first run
tracker = ClaimTracker(model="gpt-5-mini")
tracker.transform(corpus)

/opt/miniconda3/envs/traust-eval/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


10 / 12


In [23]:
# what each participant held before (pre-survey) and after the conversation
# (a constructive mediator; the greeting-only one makes no claims, so its cells below would be empty)
convo = next(c for c in corpus.iter_conversations() if c.meta["mediator"] == "constructive")
record = convo.meta["claims"]
for name in record["initial"]:
    print(f"== {name}: before")
    print("\n".join("  - " + c for c in record["initial"][name]))
    print(f"== {name}: after")
    print("\n".join("  - " + c for c in record["final"][name]))

== Goose: before
  - Data centers are the backbone of the modern digital economy.
  - Data centers are essential for technological advancement.
  - Data centers require significant resources.
  - The long-term economic benefits of building data centers outweigh their resource costs.
  - Building data centers leads to job creation in local communities.
  - Building data centers brings infrastructure improvements to local communities.
  - Local governments should prioritize investments in data centers to remain competitive in a global market.
== Goose: after
  - Data centers are the backbone of the modern digital economy and drive economic growth.
  - Data centers are essential for technological advancement.
  - Data centers require significant resources.
  - Data centers often improve long-term stability for the entire community.
  - Building data centers leads to job creation in local communities, including creation of high-tech jobs.
  - Data centers often upgrade local electrical gri

In [24]:
# everything for one conversation lives in convo.meta["claims"]
record = convo.meta["claims"]
list(record)

['speakers', 'initial', 'final', 'memory', 'steps', 'agreement']

In [25]:
# the memory: one entry per claim, for every speaker including the mediator
entry = record["memory"][0]
entry

{'id': 'c_0',
 'speaker': 'Goose',
 'role': 'participant',
 'target_speaker': 'Everyone',
 'claim': 'Data centers are the backbone of the modern digital economy and drive economic growth.',
 'kind': 'own',
 'about': None,
 'first_turn': 'pre-survey',
 'last_turn': '1',
 'status': 'active',
 'history': [{'turn': 'pre-survey',
   'op': 'ADD',
   'claim': 'Data centers are the backbone of the modern digital economy.',
   'relation': None,
   'target': None},
  {'turn': '1',
   'op': 'REFINE',
   'claim': 'Data centers are the backbone of the modern digital economy and drive economic growth.',
   'relation': 'forward_entail',
   'from': 'Data centers drive economic growth.'}],
 'restated': [],
 'links': [],
 'supersedes': None,
 'superseded_by': None,
 'retracted_at': None}

In [29]:
# the mediator's claims, with the turn they were first said at
for m in record["memory"]:
    if m["role"] == "mediator" and m["kind"] == "own":
        print(f"turn {m['first_turn']:>3}: {m['claim'][:100]}")

turn   4: An alternative is that the scale of data centers' resource demand is inherently too high to be made 
turn   8: Developers can potentially guarantee that grid upgrades and renewable projects benefit residents bef
turn  16: Specific enforcement mechanisms or independent oversight bodies are needed to ensure legally guarant


In [30]:
# a participant's claims that were linked to a mediator claim (the mediator said it first)
name = list(record["initial"])[0]
for m in record["memory"]:
    if m["speaker"] == name and any(l["speaker"] == "Mediator" for l in m["links"]):
        print(f"turn {m['first_turn']:>3}: {m['claim'][:100]}")

turn   9: Governments can mandate binding community benefit agreements to ensure grid upgrades and renewable i
turn  15: Residential priority can be legally guaranteed while using tiered agreements.
turn  18: Independent oversight and real-time data transparency build community trust.


In [31]:
# agreement between the two participants, before and after
{k: v for k, v in record["agreement"].items() if k != "pairs"}

{'initial': 0.3571428571428571,
 'final': 0.6964285714285714,
 'gain': 0.5277777777777778,
 'contradiction_initial': 0.5,
 'contradiction_final': 0.20714285714285713}

In [32]:
# per-participant numbers are stored on the speaker, keyed by conversation
speaker_id = record["speakers"][name]
corpus.get_speaker_convo_info(speaker_id, convo.id)

{'claims_initial': ['Data centers are the backbone of the modern digital economy.',
  'Data centers are essential for technological advancement.',
  'Data centers require significant resources.',
  'The long-term economic benefits of building data centers outweigh their resource costs.',
  'Building data centers leads to job creation in local communities.',
  'Building data centers brings infrastructure improvements to local communities.',
  'Local governments should prioritize investments in data centers to remain competitive in a global market.'],
 'claims_final': ['Data centers are the backbone of the modern digital economy and drive economic growth.',
  'Data centers are essential for technological advancement.',
  'Data centers require significant resources.',
  'Data centers often improve long-term stability for the entire community.',
  'Building data centers leads to job creation in local communities, including creation of high-tech jobs.',
  'Data centers often upgrade local e